# Data Modules

> datasets designed to work specifically with VAE models.

In [ ]:
#| default_exp data.data_module

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import os

import numpy as np
import pandas as pd
from omegaconf import OmegaConf
import torch
from torch.utils.data import DataLoader
import torchvision.transforms.v2 as v2
import torch.nn.functional as F

from c3jepa_wm.data.datasets import MultiAgentPOVDataset
from c3jepa_wm.data.transforms import get_transforms

## Data Modules

In [ ]:
#| export
class DataModule:
    def __init__(self,
                 data_dir: str, 
                 train_batch_size: int = 64, 
                 val_batch_size: int = 64,
                 shuffle: bool = True, 
                 num_workers: int = 0, 
                 pin_memory: bool = False,
                 drop_last: bool = False,
                 persistent_workers: bool = False):
        
        self.data_dir = data_dir
        self.train_batch_size = train_batch_size
        self.val_batch_size = val_batch_size
        self.shuffle = shuffle
        self.num_workers = num_workers
        self.pin_memory = pin_memory
        self.drop_last = drop_last
        self.persistent_workers = persistent_workers

    def setup(self):
        """Load datasets and apply transforms."""
        raise NotImplementedError("Subclasses should implement this method to load datasets.")


    def train_dataloader(self):
        """Return the training dataloader."""
        return DataLoader(
            self.train_dataset, 
            batch_size=self.train_batch_size, 
            shuffle=self.shuffle, 
            num_workers=self.num_workers, 
            pin_memory=self.pin_memory,
            sampler=self.sampler(self.train_dataset),
            collate_fn=self.collator(),
            drop_last=self.drop_last,
            persistent_workers=self.persistent_workers
        )
    
    def val_dataloader(self):
        """Return the validation dataloader."""
        return DataLoader(
            self.val_dataset, 
            batch_size=self.val_batch_size, 
            shuffle=False, 
            num_workers=self.num_workers, 
            pin_memory=self.pin_memory,
            sampler=self.sampler(self.val_dataset),
            collate_fn=self.collator(),
            drop_last=self.drop_last,
            persistent_workers=self.persistent_workers
        )
    
    def test_dataloader(self):
        """Return the test dataloader (optional)."""
        return None

    def sampler(self, dataset):
        """Return a sampler for the given dataset (optional)."""
        dist_sampler = torch.utils.data.distributed.DistributedSampler(dataset) if torch.distributed.is_initialized() else None
        return dist_sampler
    

    def collator(self):
        """Custom collate function to handle variable-length sequences (optional)."""
        colate_fn= torch.utils.data.default_collate if torch.distributed.is_initialized() else None
        return colate_fn
    

In [ ]:
#| export
class VQDataModule(DataModule):
    def __init__(self, cfg):
        self.cfg = cfg
        cfg.dataset.data_dir = self.get_data_path()
        data_params = OmegaConf.to_container(cfg.dataset, resolve=True)
        super().__init__(**data_params)
        
    def get_data_path(self):
        paths = {
            "kaggle": "/kaggle/input/datasets/ahmedkhaledsal/findgoal/",
            "puhti": "/scratch/project_2009050/datasets/findgoal",
            "mahti": "/scratch/project_2009050/datasets/findgoal/",
            "local": "/home/ahmed/Ahmed-home/1- Projects/Research/Journal 2/code/c3jepa-wm/mains/data",
        }

        for hostname, path in paths.items():
            if os.path.exists(path):
                print(f"Data path found for hostname: {hostname}")
                return path
        raise FileNotFoundError("No valid data path found for this hostname.")

    def setup(self):
        self.train_transforms, self.val_transforms = get_transforms()
        
        self.train_dataset  = MultiAgentPOVDataset(
            h5_path= os.path.join(self.cfg.dataset.data_dir, "dataset.h5"), split= "train", transform= self.train_transforms
        )

        self.val_dataset  = MultiAgentPOVDataset(
            h5_path= os.path.join(self.cfg.dataset.data_dir, "dataset.h5"), split= "val", transform= self.val_transforms
        )
    


/home/ahmed/.local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()